# Section 03: Innovating with Google Cloud AI

**Companion notebook for**: `03-ai-ml.html`

## Learning Objectives
- Call Google Cloud pre-trained ML APIs (Vision, NLP, Translation)
- Understand AutoML workflow and SDK patterns
- Build ML models with BigQuery ML using SQL
- Explore Vertex AI platform components
- Work with the Gemini API via Vertex AI

## Prerequisites
- A Google Cloud project with billing enabled (optional — mock outputs provided)
- Python 3.10+

---
## Setup & Dependencies

In [ ]:
%pip install -q google-cloud-aiplatform google-cloud-vision google-cloud-language google-cloud-translate google-cloud-bigquery vertexai

In [ ]:
import os
import json

PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "your-project-id")
LOCATION = "us-central1"

print(f"Project ID: {PROJECT_ID}")
print(f"Location:   {LOCATION}")

if PROJECT_ID == "your-project-id":
    print("WARNING: Mock outputs will be used. Set GCP_PROJECT_ID for live API calls.")

---
## Section 1: The ML Spectrum on Google Cloud

Google Cloud offers ML at every level of expertise, from zero-code
to full custom training.

In [ ]:
# ML spectrum decision guide
ml_spectrum = [
    {
        "level": "1. Pre-trained APIs",
        "expertise": "None",
        "data_needed": "None",
        "services": "Vision API, NL API, Speech, Translation, Video AI",
        "example": "Detect labels in photos, analyze sentiment in reviews",
    },
    {
        "level": "2. BigQuery ML",
        "expertise": "SQL only",
        "data_needed": "In BigQuery",
        "services": "BigQuery ML (CREATE MODEL)",
        "example": "Predict churn from customer data using SQL",
    },
    {
        "level": "3. AutoML",
        "expertise": "Label data",
        "data_needed": "Your labeled dataset",
        "services": "Vertex AI AutoML",
        "example": "Classify manufacturing defects from photos",
    },
    {
        "level": "4. Custom Training",
        "expertise": "ML engineering",
        "data_needed": "Your data + code",
        "services": "Vertex AI Custom Training",
        "example": "Train a custom transformer on proprietary data",
    },
]

print("Google Cloud ML Spectrum")
print("=" * 80)
for item in ml_spectrum:
    print(f"\n{item['level']}")
    print(f"  Expertise needed: {item['expertise']}")
    print(f"  Data needed:      {item['data_needed']}")
    print(f"  Services:         {item['services']}")
    print(f"  Example:          {item['example']}")

print("\nExam rule: Always choose the SIMPLEST option that meets requirements.")

---
## Section 2: Pre-trained APIs — Cloud Vision

The Vision API analyzes images without any training. Just send an image
and get back labels, text, faces, and more.

In [ ]:
# Call Cloud Vision API for label detection
try:
    from google.cloud import vision
    client = vision.ImageAnnotatorClient()
    
    image = vision.Image()
    image.source.image_uri = "https://storage.googleapis.com/cloud-samples-data/vision/label/wakeupcat.jpg"
    
    response = client.label_detection(image=image, max_results=8)
    
    print("Cloud Vision API — Label Detection")
    print("=" * 45)
    print(f"{'Label':<25} {'Confidence':>10}")
    print("-" * 37)
    for label in response.label_annotations:
        bar = '#' * int(label.score * 20)
        print(f"{label.description:<25} {label.score:>8.1%}  {bar}")

except Exception as e:
    print(f"Vision API failed: {e}")
    print("\nMock output — Vision API Label Detection:")
    mock_labels = [
        ("Cat", 0.983), ("Felidae", 0.961), ("Whiskers", 0.932),
        ("Carnivore", 0.891), ("Tabby cat", 0.867), ("Fur", 0.821),
        ("Snout", 0.798), ("Small to medium-sized cats", 0.776),
    ]
    print(f"\n{'Label':<30} {'Confidence':>10}")
    print("-" * 42)
    for desc, score in mock_labels:
        bar = '#' * int(score * 20)
        print(f"{desc:<30} {score:>8.1%}  {bar}")

---
## Section 3: Pre-trained APIs — Natural Language

The NL API analyzes text for sentiment, entities, and syntax.

In [ ]:
# Cloud Natural Language API — Sentiment Analysis
try:
    from google.cloud import language_v1
    client = language_v1.LanguageServiceClient()
    
    texts = [
        "Google Cloud BigQuery is amazing for data analytics!",
        "The service was terrible and the support was unhelpful.",
        "The product works fine. Nothing special.",
    ]
    
    for text in texts:
        doc = language_v1.Document(content=text, type_=language_v1.Document.Type.PLAIN_TEXT)
        result = client.analyze_sentiment(document=doc)
        s = result.document_sentiment
        label = "Positive" if s.score > 0.2 else "Negative" if s.score < -0.2 else "Neutral"
        print(f"{text[:50]:<52} score={s.score:>6.2f}  mag={s.magnitude:>5.2f}  {label}")

except Exception as e:
    print(f"NL API failed: {e}")
    print("\nMock output — Sentiment Analysis:")
    mock = [
        ("Google Cloud BigQuery is amazing for data analytics", 0.90, 0.90, "Positive"),
        ("The service was terrible and the support was unhelpf", -0.80, 1.60, "Negative"),
        ("The product works fine. Nothing special.", 0.10, 0.40, "Neutral"),
    ]
    for text, score, mag, label in mock:
        print(f"{text:<52} score={score:>6.2f}  mag={mag:>5.2f}  {label}")

---
## Section 4: BigQuery ML — ML with SQL

BigQuery ML lets SQL analysts create ML models without Python.
The following shows the SQL patterns (run in BigQuery console or client).

In [ ]:
# BigQuery ML model types and SQL patterns
bqml_models = [
    {
        "type": "Linear Regression",
        "keyword": "LINEAR_REG",
        "use_case": "Predict numeric values (price, revenue)",
        "sql": "CREATE MODEL my_dataset.price_model OPTIONS(model_type='LINEAR_REG', input_label_cols=['price']) AS SELECT ...",
    },
    {
        "type": "Logistic Regression",
        "keyword": "LOGISTIC_REG",
        "use_case": "Binary/multiclass classification",
        "sql": "CREATE MODEL my_dataset.churn_model OPTIONS(model_type='LOGISTIC_REG', input_label_cols=['churned']) AS SELECT ...",
    },
    {
        "type": "K-Means Clustering",
        "keyword": "KMEANS",
        "use_case": "Customer segmentation (unsupervised)",
        "sql": "CREATE MODEL my_dataset.segments OPTIONS(model_type='KMEANS', num_clusters=5) AS SELECT ...",
    },
    {
        "type": "Time Series (ARIMA+)",
        "keyword": "ARIMA_PLUS",
        "use_case": "Forecasting (sales, demand)",
        "sql": "CREATE MODEL my_dataset.forecast OPTIONS(model_type='ARIMA_PLUS', time_series_timestamp_col='date', time_series_data_col='sales') AS SELECT ...",
    },
    {
        "type": "XGBoost",
        "keyword": "BOOSTED_TREE_CLASSIFIER",
        "use_case": "High-accuracy classification",
        "sql": "CREATE MODEL my_dataset.xgb_model OPTIONS(model_type='BOOSTED_TREE_CLASSIFIER', input_label_cols=['target']) AS SELECT ...",
    },
]

print("BigQuery ML Model Types")
print("=" * 70)
for m in bqml_models:
    print(f"\n{m['type']} ({m['keyword']})")
    print(f"  Use case: {m['use_case']}")
    print(f"  SQL:      {m['sql'][:80]}...")

print("\nKey BQML SQL commands:")
print("  CREATE MODEL  — train a model")
print("  ML.EVALUATE   — evaluate model metrics")
print("  ML.PREDICT    — make predictions")
print("  ML.EXPLAIN_PREDICT — predictions with feature importance")

---
## Section 5: Vertex AI Platform Components

Vertex AI is the unified ML platform that brings together all ML tools.

In [ ]:
# Vertex AI components overview
vertex_components = [
    ("Model Garden", "Discover and deploy foundation models (Gemini, Llama, Mistral)"),
    ("Generative AI Studio", "Prompt design, tuning, and testing for generative AI models"),
    ("AutoML", "Train custom models with labeled data, no coding required"),
    ("Custom Training", "Full-control training with TensorFlow, PyTorch, or any framework"),
    ("Workbench", "Managed Jupyter notebooks for data science"),
    ("Feature Store", "Centralized feature repository, prevents training-serving skew"),
    ("Pipelines", "Orchestrate ML workflows as DAGs (Kubeflow/TFX)"),
    ("Endpoints", "Deploy models for online (real-time) or batch predictions"),
    ("Model Monitoring", "Detect data drift and prediction drift in production"),
    ("Experiments", "Track and compare model training runs"),
    ("Model Registry", "Version and manage trained models with metadata"),
    ("Matching Engine", "Vector similarity search for recommendation systems"),
]

print("Vertex AI Platform Components")
print("=" * 70)
for name, desc in vertex_components:
    print(f"  {name:<25} {desc}")

print("\nExam keyword: 'unified ML platform' = Vertex AI")

---
## Section 6: Gemini API via Vertex AI

Gemini is Google's most capable multimodal model, accessible through
the Vertex AI SDK.

In [ ]:
# Call Gemini via Vertex AI
try:
    import vertexai
    from vertexai.generative_models import GenerativeModel, GenerationConfig
    
    vertexai.init(project=PROJECT_ID, location=LOCATION)
    model = GenerativeModel("gemini-1.5-flash")
    
    prompt = """For the Google Cloud Digital Leader exam, explain the difference between 
    IaaS, PaaS, and SaaS in exactly 3 bullet points. Be concise."""
    
    response = model.generate_content(
        prompt,
        generation_config=GenerationConfig(temperature=0.2, max_output_tokens=300),
    )
    print("Gemini Response:")
    print(response.text)

except Exception as e:
    print(f"Gemini API failed: {e}")
    print("\nMock response:")
    print("""    
    - IaaS (Infrastructure as a Service): Provides raw compute, storage, and 
      networking. You manage OS, apps, and data. Example: Compute Engine.
    
    - PaaS (Platform as a Service): Provides a managed runtime. You deploy code,
      Google manages OS, scaling, and infrastructure. Example: App Engine, Cloud Run.
    
    - SaaS (Software as a Service): Complete applications delivered via browser.
      You manage data and access only. Example: Google Workspace, Looker.
    """)

---
## Section 7: Responsible AI Principles

Google's AI Principles guide the ethical development and deployment of AI.

In [ ]:
# Google AI Principles — study reference
principles = [
    ("1. Be socially beneficial", "AI should benefit society broadly and consider impacts."),
    ("2. Avoid unfair bias", "Test for and mitigate bias in data and model outputs."),
    ("3. Be built for safety", "Rigorous testing, monitoring, and fail-safes."),
    ("4. Be accountable to people", "Human oversight, feedback mechanisms, and control."),
    ("5. Privacy design principles", "Data minimization, transparency, and user control."),
    ("6. Scientific excellence", "Peer review, reproducibility, and open research."),
    ("7. Available for good uses", "Restrict applications that cause harm."),
]

will_not_pursue = [
    "Technologies that cause overall harm",
    "Weapons or technologies designed to injure people",
    "Surveillance violating international norms",
    "Technologies violating human rights principles",
]

print("Google's 7 AI Principles")
print("=" * 60)
for name, desc in principles:
    print(f"  {name}")
    print(f"    {desc}")

print(f"\nAI Applications Google Will NOT Pursue:")
for item in will_not_pursue:
    print(f"  - {item}")

---
## Summary

In this notebook we covered:

1. **ML Spectrum** — Pre-trained APIs (no expertise) to Custom Training (high expertise).
2. **Cloud Vision API** — Label detection, OCR, face detection from images.
3. **Cloud NL API** — Sentiment analysis, entity recognition from text.
4. **BigQuery ML** — CREATE MODEL with SQL for classification, regression, clustering, forecasting.
5. **Vertex AI** — Unified platform: Model Garden, AutoML, Pipelines, Endpoints, Monitoring.
6. **Gemini** — Multimodal generative AI model via vertexai SDK.
7. **Responsible AI** — 7 principles guiding ethical AI development.

**Next notebook**: Section 04 covers infrastructure modernization — Compute Engine, GKE, Cloud Run, and Anthos.